# LeetCode 1950 — Maximum of Minimum Values in All Subarrays bằng Python

Link đề: https://leetcode.com/problems/maximum-of-minimum-values-in-all-subarrays/ · Độ khó: **Medium** · Chủ đề: Array, Stack, Monotonic Stack

> 🔒 **Bài Premium.** Đề trong notebook được viết lại theo hiểu biết về bài (nguồn: kho lời giải doocs/leetcode). Nếu bạn có tài khoản Premium, nên đối chiếu lại câu chữ và ràng buộc với đề gốc.

Skeleton LeetCode:

```python
class Solution:
    def findMaximums(self, nums: List[int]) -> List[int]:
        ...
```

Lộ trình: **mọi độ dài × mọi mảng con × tính min** `O(n³)` → **mọi độ dài × cửa sổ trượt min bằng deque** `O(n²)` → **ngăn xếp đơn điệu: mỗi phần tử là min của cửa sổ dài nhất bao nhiêu, rồi lan kết quả từ phải sang trái** `O(n)` (lời giải nộp) → **xét giá trị giảm dần + hợp nhất đoạn (DSU)** `O(n log n)`.

## 1. Mục tiêu học

1. Dùng **ngăn xếp đơn điệu** để tìm, với mỗi `nums[i]`, phần tử **nhỏ hơn gần nhất** bên trái và bên phải ⇒ cửa sổ **dài nhất** mà `nums[i]` là min.
2. Đổi góc nhìn: thay vì "với mỗi độ dài, tìm max của các min", hỏi "**mỗi phần tử** làm min được cho những độ dài nào".
3. Nhận ra tính **đơn điệu** của đáp án: `ans[k] ≥ ans[k+1]` ⇒ lan giá trị từ độ dài lớn sang nhỏ.
4. Xử lý **phần tử bằng nhau** đúng cách (một phía nghiêm ngặt, một phía không — hoặc cả hai nghiêm ngặt).
5. Kiểm chứng bằng vét cạn ngẫu nhiên có seed; so với cách DSU.

## 2. Đề bài & ví dụ

Cho mảng `nums` độ dài `n`. Trả lời `n` truy vấn, với mỗi `i` trong `0..n−1`:
- Tìm **giá trị nhỏ nhất** của mỗi mảng con độ dài `i + 1`.
- Lấy **giá trị lớn nhất** trong các giá trị nhỏ nhất đó — đó là `ans[i]`.

**Ví dụ 1:** `nums = [0,1,2,4]` → `[4,2,1,0]`.

**Ví dụ 2:** `nums = [10,20,50,10]` → `[50,20,10,10]`.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `0 ≤ nums[i] ≤ 10⁹`.

In [1]:
import random
import time
from collections import deque
from typing import List

examples = [([0, 1, 2, 4], [4, 2, 1, 0]), ([10, 20, 50, 10], [50, 20, 10, 10])]
for nums, want in examples:
    print(nums, "->", want)

[0, 1, 2, 4] -> [4, 2, 1, 0]
[10, 20, 50, 10] -> [50, 20, 10, 10]


## 3. Kiến thức nền

### 3.1 Ngăn xếp đơn điệu tăng: phần tử nhỏ hơn gần nhất bên trái

In [2]:
def prev_smaller(nums):
    left, st = [-1] * len(nums), []
    for i, x in enumerate(nums):
        while st and nums[st[-1]] >= x:          # bỏ các phần tử >= x
            st.pop()
        left[i] = st[-1] if st else -1
        st.append(i)
    return left

print(prev_smaller([10, 20, 50, 10]))

[-1, 0, 1, -1]


Mỗi chỉ số vào/ra ngăn xếp một lần ⇒ `O(n)`. Phía phải làm đối xứng (duyệt từ phải, hoặc dùng lúc `pop`).

### 3.2 Cửa sổ dài nhất mà `nums[i]` là min

Nếu `L[i]` = chỉ số phần tử **nhỏ hơn** gần nhất bên trái, `R[i]` = bên phải, thì mọi mảng con nằm trong `(L[i], R[i])` và chứa `i` có min `= nums[i]` (khi không có phần tử bằng nhau gây nhiễu). Độ dài tối đa `R[i] − L[i] − 1`.

In [3]:
nums = [10, 20, 50, 10]
L = prev_smaller(nums)
R = [len(nums)] * len(nums); st = []
for i in range(len(nums) - 1, -1, -1):
    while st and nums[st[-1]] >= nums[i]: st.pop()
    R[i] = st[-1] if st else len(nums); st.append(i)
print("L =", L, "R =", R, "| độ dài cửa sổ =", [R[i] - L[i] - 1 for i in range(len(nums))])

L = [-1, 0, 1, -1] R = [4, 3, 3, 4] | độ dài cửa sổ = [4, 2, 1, 4]


### 3.3 Cửa sổ trượt min bằng deque

Deque lưu chỉ số với giá trị **tăng dần**; đầu deque là min của cửa sổ hiện tại. Mỗi phần tử vào/ra một lần ⇒ `O(n)` cho một độ dài.

In [4]:
def window_mins(nums, k):
    dq, out = deque(), []
    for i, x in enumerate(nums):
        while dq and nums[dq[-1]] >= x: dq.pop()
        dq.append(i)
        if dq[0] <= i - k: dq.popleft()
        if i >= k - 1: out.append(nums[dq[0]])
    return out

print(window_mins([10, 20, 50, 10], 2))

[10, 20, 10]


### 3.4 Lan giá trị tối đa từ phải sang trái

`ans[k] = max(ans[k], ans[k+1])`: nếu một giá trị là min của cửa sổ dài `k+2` thì cửa sổ con dài `k+1` có min **≥** giá trị đó.

In [5]:
ans = [5, 0, 3, 0]
for k in range(len(ans) - 2, -1, -1):
    ans[k] = max(ans[k], ans[k + 1])
print(ans)

[5, 3, 3, 0]


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 2 `nums = [10,20,50,10]`. Với mỗi phần tử, cửa sổ **dài nhất** mà nó là min:

| i | `nums[i]` | nhỏ hơn gần nhất trái / phải | cửa sổ | độ dài |
|---|---|---|---|---|
| 0 | 10 | — / — (không có số **nhỏ hơn**) | `[0..3]` | 4 |
| 1 | 20 | 0 / 3 | `[1..2]` | 2 |
| 2 | 50 | 1 / 3 | `[2..2]` | 1 |
| 3 | 10 | — / — | `[0..3]` | 4 |

Ghi: độ dài 4 ← 10, độ dài 2 ← 20, độ dài 1 ← 50. Độ dài 3 chưa có ai ⇒ lấy từ độ dài lớn hơn (4): 10. Kết quả `[50, 20, 10, 10]`. ✔

> Ghi chú: bảng trên dùng biên **nhỏ hơn ngặt ở cả hai phía** (như code ở 3.2). Lời giải nộp (mục 7.2, 9) bật bằng `≥` nên biên phải là phần tử **≤** gần nhất: số 10 ở `i=0` bị chặn bởi số 10 ở `i=3` và chỉ nhận độ dài 3, còn số 10 ở `i=3` vẫn nhận độ dài 4. Cả hai quy ước đều cho cùng đáp án (chứng minh ở mục 10).

Vì sao lan được? Nếu `x` là min của một cửa sổ dài `L`, thì mọi cửa sổ con dài `< L` nằm trong nó có min `≥ x` ⇒ `ans[độ dài nhỏ hơn] ≥ x`.

## 5. Cách 1 — Brute Force / baseline: mọi độ dài, mọi mảng con

In [6]:
def find_max_brute(nums):
    n = len(nums)
    return [max(min(nums[l:l + k]) for l in range(n - k + 1)) for k in range(1, n + 1)]

for nums, want in examples:
    print(find_max_brute(nums), "| kỳ vọng", want)

[4, 2, 1, 0] | kỳ vọng [4, 2, 1, 0]
[50, 20, 10, 10] | kỳ vọng [50, 20, 10, 10]


**Độ phức tạp:** `Σ_k (n − k + 1) · k = O(n³)`. **Nút thắt:** tính min lại từ đầu cho mỗi cửa sổ; cửa sổ trượt chỉ thay đổi hai phần tử.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (đảo câu hỏi):** với mỗi `i`, gọi `len_i` = độ dài cửa sổ dài nhất có min `= nums[i]` (giới hạn bởi phần tử **nhỏ hơn ngặt** gần nhất hai bên). Khi đó `nums[i]` là ứng viên cho **mọi** độ dài `1..len_i`.

**Insight 2 (đơn điệu):** `ans[k−1] = max{ nums[i] : len_i ≥ k }`. Tính bằng: `best[len_i] = max(best[len_i], nums[i])`, rồi lan `best[k] = max(best[k], best[k+1])` từ `k = n` xuống `1`.

**Quy tắc ngăn xếp:** duyệt trái sang phải, giữ ngăn xếp chỉ số với giá trị **tăng ngặt**; khi gặp `x`, mọi phần tử đỉnh `≥ x` bị bật — với phần tử bị bật `j`, biên phải là `i` (phần tử đầu tiên bên phải có giá trị `≤ nums[j]`), biên trái là đỉnh mới (phần tử gần nhất bên trái có giá trị `< nums[j]`). (Ở đây dùng `≥` khi bật ⇒ với giá trị bằng nhau, phần tử trước bị chặn bởi phần tử sau — nhưng phần tử **sau** sẽ có cửa sổ đủ rộng, nên đáp án không sai.)

**Invariant ngăn xếp:** giá trị tại các chỉ số trong ngăn xếp tăng ngặt từ đáy lên đỉnh.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mỗi độ dài một lượt cửa sổ trượt (`O(n²)`)

In [7]:
def find_max_window(nums):
    n = len(nums)
    return [max(window_mins(nums, k)) for k in range(1, n + 1)]

for nums, want in examples:
    print(find_max_window(nums), want)

[4, 2, 1, 0] [4, 2, 1, 0]
[50, 20, 10, 10] [50, 20, 10, 10]


**Độ phức tạp:** `n` độ dài × `O(n)` ⇒ `O(n²)` — `10¹⁰` với `n = 10⁵`. Nút thắt: tính lại cho từng độ dài thay vì để mỗi phần tử "đóng góp" một lần.

### 7.2 Cách 3 — Ngăn xếp đơn điệu + lan kết quả (`O(n)`) — lời giải chuẩn

In [8]:
def find_max_stack_demo(nums):
    n = len(nums)
    left = [-1] * n; right = [n] * n; st = []
    for i, x in enumerate(nums):
        while st and nums[st[-1]] >= x:
            right[st.pop()] = i                 # phần tử bị bật có biên phải = i
        left[i] = st[-1] if st else -1
        st.append(i)
    best = [0] * (n + 1)
    for i in range(n):
        L = right[i] - left[i] - 1
        best[L] = max(best[L], nums[i])
        print(f"i={i}: nums={nums[i]:>3}, cửa sổ ({left[i]}, {right[i]}) dài {L}")
    print("best theo độ dài (trước khi lan):", best[1:])
    for k in range(n - 1, 0, -1):
        best[k] = max(best[k], best[k + 1])
    print("sau khi lan:", best[1:])
    return best[1:]

print("=>", find_max_stack_demo([10, 20, 50, 10]))

i=0: nums= 10, cửa sổ (-1, 3) dài 3
i=1: nums= 20, cửa sổ (0, 3) dài 2
i=2: nums= 50, cửa sổ (1, 3) dài 1
i=3: nums= 10, cửa sổ (-1, 4) dài 4
best theo độ dài (trước khi lan): [50, 20, 10, 10]
sau khi lan: [50, 20, 10, 10]
=> [50, 20, 10, 10]


In [9]:
class SolutionStack:
    def findMaximums(self, nums: List[int]) -> List[int]:
        n = len(nums)
        left = [-1] * n; right = [n] * n; st = []
        for i, x in enumerate(nums):
            while st and nums[st[-1]] >= x:
                right[st.pop()] = i
            left[i] = st[-1] if st else -1
            st.append(i)
        best = [0] * (n + 2)
        for i in range(n):
            L = right[i] - left[i] - 1
            if nums[i] > best[L]:
                best[L] = nums[i]
        for k in range(n - 1, 0, -1):
            if best[k + 1] > best[k]:
                best[k] = best[k + 1]
        return best[1:n + 1]

for nums, want in examples:
    print(SolutionStack().findMaximums(nums) == want)

True
True


Về phần tử **bằng nhau**: với `≥` khi bật, phần tử `j` bị bật bởi `i` có `nums[i] = nums[j]` ⇒ `right[j] = i` hơi "ngắn". Nhưng phần tử **cuối cùng** trong nhóm bằng nhau liên tiếp (theo ngăn xếp) có `left` vượt qua các phần tử bằng nó (vì chúng đã bị bật) và `right` là phần tử **nhỏ hơn** thật ⇒ nhận đúng độ dài tối đa. Kiểm ở 12.1 với `[5, 5]`, `[3,3,3]`.

**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ.

### 7.3 Cách 4 — Giá trị giảm dần + hợp nhất đoạn (DSU) `O(n log n)`

Xét phần tử theo giá trị **giảm dần**; "bật" vị trí `i` và hợp nhất với các hàng xóm đã bật. Khi bật `nums[i]`, đoạn liên tiếp chứa `i` gồm toàn phần tử `≥ nums[i]`, độ dài `s` ⇒ mọi độ dài `≤ s` có `ans ≥ nums[i]`. Vì xét giá trị giảm dần, lần **đầu tiên** một độ dài `k` được phủ cho đúng `ans[k−1]`.

In [10]:
class SolutionDSU:
    def findMaximums(self, nums: List[int]) -> List[int]:
        n = len(nums)
        parent = list(range(n)); size = [1] * n; on = [False] * n
        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]; x = parent[x]
            return x
        def union(a, b):
            a, b = find(a), find(b)
            if a == b: return a
            if size[a] < size[b]: a, b = b, a
            parent[b] = a; size[a] += size[b]; return a
        ans = [0] * n
        covered = 0                                # mọi độ dài 1..covered đã có đáp án
        for i in sorted(range(n), key=lambda i: -nums[i]):
            on[i] = True
            r = i
            if i > 0 and on[i - 1]: r = union(r, i - 1)
            if i + 1 < n and on[i + 1]: r = union(r, i + 1)
            s = size[find(r)]
            while covered < s:
                ans[covered] = nums[i]; covered += 1
        return ans

for nums, want in examples:
    print(SolutionDSU().findMaximums(nums) == want)

True
True


**Độ phức tạp:** sắp xếp `O(n log n)` + DSU gần `O(n)`; con trỏ `covered` chỉ tiến ⇒ `O(n log n)` tổng.

## 8. TRACE từng bước

Trace ngăn xếp (Cách 3) trên Ví dụ 1 `[0,1,2,4]`:

| i | x | bật (gán `right`) | `left[i]` | ngăn xếp sau |
|---|---|---|---|---|
| 0 | 0 | — | −1 | `[0]` |
| 1 | 1 | — | 0 | `[0,1]` |
| 2 | 2 | — | 1 | `[0,1,2]` |
| 3 | 4 | — | 2 | `[0,1,2,3]` |

Hết mảng: mọi `right = 4`. Độ dài: `i=0: 4−(−1)−1 = 4`, `i=1: 3`, `i=2: 2`, `i=3: 1` ⇒ `best[4]=0, best[3]=1, best[2]=2, best[1]=4` ⇒ `[4,2,1,0]`. ✔

In [11]:
find_max_stack_demo([0, 1, 2, 4])

i=0: nums=  0, cửa sổ (-1, 4) dài 4
i=1: nums=  1, cửa sổ (0, 4) dài 3
i=2: nums=  2, cửa sổ (1, 4) dài 2
i=3: nums=  4, cửa sổ (2, 4) dài 1
best theo độ dài (trước khi lan): [4, 2, 1, 0]
sau khi lan: [4, 2, 1, 0]


[4, 2, 1, 0]

Trace ngăn xếp chi tiết (in trạng thái ngăn xếp sau mỗi bước) trên một mảng có giá trị trùng `[2,1,2,1,2]` — đây là chỗ dễ nhầm nhất:

In [12]:
def trace_stack(nums):
    n = len(nums); left, right, st = [-1] * n, [n] * n, []
    for i, x in enumerate(nums):
        popped = []
        while st and nums[st[-1]] >= x:
            j = st.pop(); right[j] = i; popped.append(j)
        left[i] = st[-1] if st else -1
        st.append(i)
        info = ", ".join(f"j={j} (right={i})" for j in popped) or "—"
        print(f"i={i} x={x}: bật {info:<28} left[{i}]={left[i]:>2}  ngăn xếp={[nums[k] for k in st]}")
    lens = [right[i] - left[i] - 1 for i in range(n)]
    print("độ dài cửa sổ:", lens)
    return lens

trace_stack([2, 1, 2, 1, 2])

i=0 x=2: bật —                            left[0]=-1  ngăn xếp=[2]
i=1 x=1: bật j=0 (right=1)                left[1]=-1  ngăn xếp=[1]
i=2 x=2: bật —                            left[2]= 1  ngăn xếp=[1, 2]
i=3 x=1: bật j=2 (right=3), j=1 (right=3) left[3]=-1  ngăn xếp=[1]
i=4 x=2: bật —                            left[4]= 3  ngăn xếp=[1, 2]
độ dài cửa sổ: [1, 3, 1, 5, 1]


[1, 3, 1, 5, 1]

Quan sát: số 1 ở `i=1` bị số 1 ở `i=3` bật ⇒ `right[1] = 3`, độ dài chỉ 3. Nhưng số 1 ở `i=3` có `left = −1` (vì `i=1` đã bị bật) và `right = 5` ⇒ độ dài 5 — "đại diện" cho nhóm giá trị bằng nhau.

Trace DSU trên Ví dụ 2 (giá trị giảm dần: 50, 20, 10, 10):

In [13]:
def trace_dsu(nums):
    n = len(nums); on = [False] * n; ans = [None] * n; covered = 0
    for i in sorted(range(n), key=lambda i: -nums[i]):
        on[i] = True
        l = i
        while l > 0 and on[l - 1]: l -= 1
        r = i
        while r + 1 < n and on[r + 1]: r += 1
        s = r - l + 1
        newly = list(range(covered + 1, s + 1))
        while covered < s:
            ans[covered] = nums[i]; covered += 1
        print(f"bật i={i} (giá trị {nums[i]}): đoạn [{l}..{r}] dài {s} -> các độ dài mới phủ {newly}")
    print("ans =", ans)
    return ans

trace_dsu([10, 20, 50, 10])

bật i=2 (giá trị 50): đoạn [2..2] dài 1 -> các độ dài mới phủ [1]
bật i=1 (giá trị 20): đoạn [1..2] dài 2 -> các độ dài mới phủ [2]
bật i=0 (giá trị 10): đoạn [0..2] dài 3 -> các độ dài mới phủ [3]
bật i=3 (giá trị 10): đoạn [0..3] dài 4 -> các độ dài mới phủ [4]
ans = [50, 20, 10, 10]


[50, 20, 10, 10]

## 9. Code nộp LeetCode

In [14]:
from typing import List

class Solution:
    def findMaximums(self, nums: List[int]) -> List[int]:
        n = len(nums)
        left, right, st = [-1] * n, [n] * n, []
        for i, x in enumerate(nums):
            while st and nums[st[-1]] >= x:       # ngăn xếp giá trị tăng ngặt
                right[st.pop()] = i               # biên phải của phần tử bị bật
            left[i] = st[-1] if st else -1        # biên trái: phần tử nhỏ hơn gần nhất
            st.append(i)
        best = [0] * (n + 1)                      # best[L]: max của các phần tử có cửa sổ dài L
        for i in range(n):
            L = right[i] - left[i] - 1
            best[L] = max(best[L], nums[i])
        for L in range(n - 1, 0, -1):             # cửa sổ dài hơn -> cửa sổ ngắn hơn
            best[L] = max(best[L], best[L + 1])
        return best[1:]

for nums, want in examples:
    assert Solution().findMaximums(nums) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Gọi `ans[k]` (1-indexed theo độ dài `k`) = max các min của mảng con độ dài `k`.

**(1) Mỗi phần tử là min của cửa sổ dài `len_i`.** Gọi `ℓ` = phần tử nhỏ hơn ngặt gần nhất bên trái của `i`, `ρ` = phần tử nhỏ hơn **hoặc bằng** gần nhất bên phải (theo cách bật `≥`). Mọi phần tử trong `(ℓ, ρ)` đều `≥ nums[i]` ⇒ min của cửa sổ `(ℓ, ρ)` là `nums[i]`, độ dài `len_i = ρ − ℓ − 1`. Do đó `ans[k] ≥ nums[i]` với mọi `k ≤ len_i` (lấy cửa sổ con chứa `i`).

**(2) Ngược lại, mọi cặp (độ dài `k`, min `v`) đều được "bắt".** Lấy một mảng con độ dài `k` có min `v` đạt tại vị trí **phải nhất** `i` trong mảng con có giá trị `v`. Mọi phần tử trong mảng con `≥ v`, và bên phải `i` trong mảng con không có phần tử `= v` ⇒ cửa sổ `(ℓ, ρ)` của `i` chứa trọn mảng con ⇒ `len_i ≥ k` ⇒ `best` sau khi lan tại `k` `≥ v`.

Từ (1) và (2): `ans[k] = max{nums[i] : len_i ≥ k}` — đúng là `best[k]` sau bước lan từ phải sang trái.

## 11. Phân tích Time / Space

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n³)` | `O(n)` |
| 2. Cửa sổ trượt mỗi độ dài | `O(n²)` | `O(n)` |
| 3. Ngăn xếp đơn điệu | `O(n)` | `O(n)` |
| 4. DSU theo giá trị giảm | `O(n log n)` | `O(n)` |

In [15]:
rng = random.Random(1950)
big = [rng.randint(0, 10**9) for _ in range(10**5)]
inc = list(range(10**5))
eq = [7] * 10**5
for label, data in [("ngẫu nhiên", big), ("tăng dần", inc), ("toàn bằng nhau", eq)]:
    res = []
    for name, sol in [("Ngăn xếp/Nộp", Solution()), ("DSU", SolutionDSU())]:
        t = time.perf_counter(); r = sol.findMaximums(data); res.append(tuple(r))
        print(f"[{label}] {name}: {1000*(time.perf_counter()-t):.0f} ms")
    assert len(set(res)) == 1
t = time.perf_counter(); find_max_window(big[:600]); dt = time.perf_counter() - t
print(f"Cách 2 với n=600: {dt:.2f}s -> n=10^5 ước tính ≈ {dt * (10**5 / 600) ** 2:.0f}s")

[ngẫu nhiên] Ngăn xếp/Nộp: 26 ms


[ngẫu nhiên] DSU: 101 ms


[tăng dần] Ngăn xếp/Nộp: 19 ms


[tăng dần] DSU: 47 ms
[toàn bằng nhau] Ngăn xếp/Nộp: 18 ms


[toàn bằng nhau] DSU: 45 ms
Cách 2 với n=600: 0.05s -> n=10^5 ước tính ≈ 1432s


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng, đối chiếu vét cạn)

In [16]:
sols = [Solution(), SolutionStack(), SolutionDSU()]
edge = examples + [
    ([5], [5]),
    ([5, 5], [5, 5]),
    ([3, 3, 3], [3, 3, 3]),
    ([1, 2, 1], [2, 1, 1]),
    ([4, 3, 2, 1], [4, 3, 2, 1]),
    ([2, 1, 2, 1, 2], [2, 1, 1, 1, 1]),
    ([0, 0, 10**9], [10**9, 0, 0]),
]
for nums, want in edge:
    assert find_max_brute(nums) == want, (nums, find_max_brute(nums))
    for s in sols:
        assert s.findMaximums(nums) == want, (nums, s)
    assert find_max_window(nums) == want
    print("OK", nums, "->", want)

OK [0, 1, 2, 4] -> [4, 2, 1, 0]
OK [10, 20, 50, 10] -> [50, 20, 10, 10]
OK [5] -> [5]
OK [5, 5] -> [5, 5]
OK [3, 3, 3] -> [3, 3, 3]
OK [1, 2, 1] -> [2, 1, 1]
OK [4, 3, 2, 1] -> [4, 3, 2, 1]
OK [2, 1, 2, 1, 2] -> [2, 1, 1, 1, 1]
OK [0, 0, 1000000000] -> [1000000000, 0, 0]


### 12.2 Ngẫu nhiên có seed: so với vét cạn (nhiều giá trị trùng)

In [17]:
rng = random.Random(42)
for _ in range(800):
    n = rng.randint(1, 14)
    hi = rng.choice([1, 3, 10, 10**9])
    nums = [rng.randint(0, hi) for _ in range(n)]
    want = find_max_brute(nums)
    for s in sols:
        assert s.findMaximums(nums) == want, nums
    assert find_max_window(nums) == want
print("800 bộ ngẫu nhiên: khớp vét cạn")

800 bộ ngẫu nhiên: khớp vét cạn


### 12.3 Tính chất: đáp án không tăng

In [18]:
rng = random.Random(7)
for _ in range(300):
    nums = [rng.randint(0, 50) for _ in range(rng.randint(1, 60))]
    a = Solution().findMaximums(nums)
    assert all(a[i] >= a[i + 1] for i in range(len(a) - 1)) and a[0] == max(nums) and a[-1] == min(nums)
print("ans không tăng; ans[0] = max(nums), ans[-1] = min(nums)")

ans không tăng; ans[0] = max(nums), ans[-1] = min(nums)


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên bước lan từ phải sang trái

In [19]:
def wrong_no_propagate(nums):
    n = len(nums); left, right, st = [-1] * n, [n] * n, []
    for i, x in enumerate(nums):
        while st and nums[st[-1]] >= x: right[st.pop()] = i
        left[i] = st[-1] if st else -1; st.append(i)
    best = [0] * (n + 1)
    for i in range(n):
        L = right[i] - left[i] - 1; best[L] = max(best[L], nums[i])
    return best[1:]                                          # SAI: thiếu lan

for nums in ([3, 1, 3], [2, 1, 2, 1, 2], [10, 20, 50, 10]):
    print(nums, "sai:", wrong_no_propagate(nums), "| đúng:", find_max_brute(nums))

[3, 1, 3] sai: [3, 0, 1] | đúng: [3, 1, 1]
[2, 1, 2, 1, 2] sai: [2, 0, 1, 0, 1] | đúng: [2, 1, 1, 1, 1]
[10, 20, 50, 10] sai: [50, 20, 10, 10] | đúng: [50, 20, 10, 10]


**Vì sao sai:** chỉ độ dài **tối đa** của mỗi phần tử được ghi; các độ dài ngắn hơn không có ai ghi ⇒ còn 0 (`[3,1,3]`: độ dài 2 không có phần tử nào có cửa sổ dài đúng 2). Hai ví dụ của đề tình cờ vẫn ra đúng (Ví dụ 2 đúng vì số 10 đầu tiên nhận độ dài 3) — đừng tin vào việc "chạy đúng ví dụ". Một phần tử là min của cửa sổ dài `L` cũng là ứng viên cho mọi độ dài `< L`.

### Lỗi 2 — Cả hai phía đều dừng ở phần tử **bằng** (cửa sổ quá hẹp)

In [20]:
def wrong_both_nonstrict(nums):
    n = len(nums)
    left = [-1] * n; st = []
    for i, x in enumerate(nums):
        while st and nums[st[-1]] > x: st.pop()                 # giữ phần tử bằng -> chặn bên trái
        left[i] = st[-1] if st else -1; st.append(i)
    right = [n] * n; st = []
    for i in range(n - 1, -1, -1):
        while st and nums[st[-1]] > nums[i]: st.pop()           # chặn bên phải bởi phần tử bằng
        right[i] = st[-1] if st else n; st.append(i)
    best = [0] * (n + 2)
    for i in range(n):
        L = right[i] - left[i] - 1; best[L] = max(best[L], nums[i])
    for k in range(n - 1, 0, -1): best[k] = max(best[k], best[k + 1])
    return best[1:n + 1]

for nums in ([5, 5], [3, 3, 3], [1, 2, 2, 1]):
    print(nums, "sai:", wrong_both_nonstrict(nums), "| đúng:", Solution().findMaximums(nums))

[5, 5] sai: [5, 0] | đúng: [5, 5]
[3, 3, 3] sai: [3, 0, 0] | đúng: [3, 3, 3]
[1, 2, 2, 1] sai: [2, 1, 1, 0] | đúng: [2, 2, 1, 1]


**Vì sao sai:** nếu cả hai phía coi phần tử **bằng** là "chặn", mỗi phần tử trong một dãy giá trị bằng nhau chỉ có cửa sổ dài 1 — không ai nhận độ dài lớn ⇒ `[5, 5]` cho `[5, 0]`. **Sửa:** ít nhất một phía chỉ dừng ở phần tử **nhỏ hơn ngặt** (cách bật `≥` ở lời giải làm điều đó tự động).

### Lỗi 3 — Gán `ans` theo độ dài nhưng không lấy `max` (ghi đè)

In [21]:
def wrong_overwrite(nums):
    n = len(nums); left, right, st = [-1] * n, [n] * n, []
    for i, x in enumerate(nums):
        while st and nums[st[-1]] >= x: right[st.pop()] = i
        left[i] = st[-1] if st else -1; st.append(i)
    best = [0] * (n + 1)
    for i in range(n):
        best[right[i] - left[i] - 1] = nums[i]                  # SAI: ghi đè thay vì max
    for k in range(n - 1, 0, -1): best[k] = max(best[k], best[k + 1])
    return best[1:]

for nums in ([3, 1, 2], [2, 9, 1, 8]):
    print(nums, "sai:", wrong_overwrite(nums), "| đúng:", Solution().findMaximums(nums))

[3, 1, 2] sai: [2, 1, 1] | đúng: [3, 1, 1]
[2, 9, 1, 8] sai: [8, 2, 1, 1] | đúng: [9, 2, 1, 1]


**Vì sao sai:** nhiều phần tử có thể cùng độ dài cửa sổ (`[2,9,1,8]`: `9` và `8` đều có cửa sổ dài 1). Phần tử xét sau ghi đè phần tử trước, có thể làm mất giá trị lớn hơn.

### Lỗi 4 — Dùng cách `O(n²)` cho `n = 10⁵`

In [22]:
t = time.perf_counter(); find_max_window(big[:1000]); dt = time.perf_counter() - t
print(f"cửa sổ trượt cho mọi độ dài, n=1000: {dt:.2f}s -> n=10^5 ≈ {dt * 10**4:.0f}s (TLE)")

cửa sổ trượt cho mọi độ dài, n=1000: 0.18s -> n=10^5 ≈ 1752s (TLE)


**Vì sao chưa tốt:** đúng nhưng `O(n²)`. Mỗi phần tử chỉ cần "đóng góp" một lần cho độ dài cửa sổ tối đa của nó.

### Lỗi 5 — Lệch một khi tính độ dài cửa sổ (`right − left` thay vì `right − left − 1`)

In [23]:
def wrong_off_by_one(nums):
    n = len(nums); left, right, st = [-1] * n, [n] * n, []
    for i, x in enumerate(nums):
        while st and nums[st[-1]] >= x: right[st.pop()] = i
        left[i] = st[-1] if st else -1; st.append(i)
    best = [0] * (n + 2)
    for i in range(n):
        L = min(right[i] - left[i], n)                       # SAI: thừa 1 (kẹp lại để không tràn mảng)
        best[L] = max(best[L], nums[i])
    for k in range(n - 1, 0, -1): best[k] = max(best[k], best[k + 1])
    return best[1:n + 1]

for nums in ([0, 1, 2, 4], [10, 20, 50, 10]):
    print(nums, "sai:", wrong_off_by_one(nums), "| đúng:", Solution().findMaximums(nums))

[0, 1, 2, 4] sai: [4, 4, 2, 1] | đúng: [4, 2, 1, 0]
[10, 20, 50, 10] sai: [50, 50, 20, 10] | đúng: [50, 20, 10, 10]


**Vì sao sai:** `left` và `right` là các chỉ số **bị loại** (phần tử nhỏ hơn hoặc biên ảo `−1`, `n`), nên cửa sổ là `left+1 .. right−1`, có `right − left − 1` phần tử. Thừa 1 khiến mỗi phần tử "nhận" một độ dài mà thực ra nó không là min ⇒ đáp án quá lớn. Nếu không kẹp `min(…, n)`, code còn ném `IndexError` khi `left = −1`, `right = n`. **Mẹo kiểm:** phần tử nhỏ nhất toàn mảng phải có độ dài đúng `n`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Với mỗi mảng con / mỗi độ dài, min (hoặc max) là gì"** ⇒ đảo câu hỏi: **mỗi phần tử là min của những cửa sổ nào** ⇒ ngăn xếp đơn điệu tìm biên nhỏ hơn gần nhất.
- Đáp án theo độ dài **đơn điệu** ⇒ ghi tại độ dài tối đa rồi **lan**.
- Phần tử bằng nhau: một phía `<`, một phía `≤` (hoặc bật bằng `≥` và dùng `pop` cho biên phải).
- Cách khác thường gặp: xét giá trị giảm dần + DSU nối đoạn.
- Bài cùng họ: LeetCode 84 *Largest Rectangle in Histogram*, 907 *Sum of Subarray Minimums*, 1856 *Maximum Subarray Min-Product*.

## 15. Biến thể của bài

1. **Min của các max** theo độ dài (đối ngẫu) ⇒ đổi dấu hoặc ngăn xếp giảm (Bài tập 5).
2. **Tổng các min của mọi mảng con** (LeetCode 907) ⇒ đóng góp `nums[i] · (i − left) · (right − i)`.
3. **Cửa sổ có độ dài cố định `k` duy nhất** ⇒ deque một lượt (LeetCode 239 cho max).
4. **Truy vấn độ dài tùy ý nhiều lần** ⇒ tiền xử lý như bài, trả lời `O(1)`.
5. **Tích min × tổng** (LeetCode 1856) ⇒ cùng biên ngăn xếp + tổng tiền tố.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | mọi mảng con | `O(n³)` | `O(n)` | kiểm chứng |
| 2. Cửa sổ trượt | deque cho từng độ dài | `O(n²)` | `O(n)` | `n` ≤ vài nghìn |
| 3. Ngăn xếp đơn điệu | cửa sổ dài nhất mỗi phần tử + lan | `O(n)` | `O(n)` | **nộp bài** |
| 4. DSU giá trị giảm | bật dần, nối đoạn | `O(n log n)` | `O(n)` | trực quan "mực nước hạ dần" |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `[5]`, `[1,2,1]`, `[3,1,3]`?

**Bài 2 (tính tay):** Tính `left`, `right`, độ dài cửa sổ và `best` (trước/sau khi lan) cho `[2,1,2,1,2]`.

**Bài 3 (điền TODO):** Viết `window_lengths(nums)` trả về độ dài cửa sổ dài nhất mà `nums[i]` là min, cho mọi `i`.

**Bài 4 (tự viết lại):** Viết `sum_subarray_mins(nums)` (LeetCode 907, không modulo) bằng cùng biên ngăn xếp.

**Bài 5 (mini challenge):** Viết `find_minimums_of_max(nums)`: với mỗi độ dài `k`, **min** của các **max** của mảng con độ dài `k`.

In [24]:
def window_lengths(nums):
    # TODO
    pass

def sum_subarray_mins(nums):
    # TODO
    pass

def find_minimums_of_max(nums):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | `[5] → [5]` | `[1,2,1] → [2,1,1]` | `[3,1,3] → [3,1,1]` |
| 2 | Các số 2 có cửa sổ dài 1 | Theo cách bật `≥`: số 1 ở `i=1` bị chặn bởi số 1 ở `i=3` ⇒ dài 3; số 1 ở `i=3` dài 5 | `best` trước lan (độ dài 1..5): `[2, 0, 1, 0, 1]`; sau lan `[2, 1, 1, 1, 1]` (nếu cả hai phía ngặt: trước lan `[2, 0, 0, 0, 1]`) |
| 3 | `right[i] − left[i] − 1` | Dùng đúng cách bật `≥` | Kiểm bằng vét cạn với định nghĩa "min đạt tại vị trí phải nhất" |
| 4 | Đóng góp `nums[i] · (i − left[i]) · (right[i] − i)` | Một phía `<`, một phía `≤` để không đếm trùng | So vét cạn |
| 5 | Đối ngẫu: biên là phần tử **lớn hơn** gần nhất (ngăn xếp giảm) | `best[L] = min(...)`, lan bằng `min` | Khởi tạo `best` bằng `+∞` — nếu đổi dấu rồi gọi lời giải gốc thì `best = 0` sẽ sai với số âm |

## 19. Đáp án tham khảo

In [25]:
r1 = [Solution().findMaximums(a) for a in ([5], [1, 2, 1], [3, 1, 3])]
print("Bài 1:", r1); assert r1 == [[5], [2, 1, 1], [3, 1, 1]]
print("Bài 2:"); assert find_max_stack_demo([2, 1, 2, 1, 2]) == [2, 1, 1, 1, 1]

Bài 1: [[5], [2, 1, 1], [3, 1, 1]]
Bài 2:
i=0: nums=  2, cửa sổ (-1, 1) dài 1
i=1: nums=  1, cửa sổ (-1, 3) dài 3
i=2: nums=  2, cửa sổ (1, 3) dài 1
i=3: nums=  1, cửa sổ (-1, 5) dài 5
i=4: nums=  2, cửa sổ (3, 5) dài 1
best theo độ dài (trước khi lan): [2, 0, 1, 0, 1]
sau khi lan: [2, 1, 1, 1, 1]


In [26]:
def window_lengths(nums):
    n = len(nums); left, right, st = [-1] * n, [n] * n, []
    for i, x in enumerate(nums):
        while st and nums[st[-1]] >= x: right[st.pop()] = i
        left[i] = st[-1] if st else -1; st.append(i)
    return [right[i] - left[i] - 1 for i in range(n)]

def sum_subarray_mins(nums):
    n = len(nums); left, right, st = [-1] * n, [n] * n, []
    for i, x in enumerate(nums):
        while st and nums[st[-1]] >= x: right[st.pop()] = i
        left[i] = st[-1] if st else -1; st.append(i)
    return sum(nums[i] * (i - left[i]) * (right[i] - i) for i in range(n))

assert sum_subarray_mins([3, 1, 2, 4]) == 17 and sum_subarray_mins([11, 81, 94, 43, 3]) == 444   # ví dụ LeetCode 907
rng = random.Random(3)
for _ in range(400):
    nums = [rng.randint(0, 6) for _ in range(rng.randint(1, 12))]
    n = len(nums)
    assert sum_subarray_mins(nums) == sum(min(nums[l:r + 1]) for l in range(n) for r in range(l, n))
    wl = window_lengths(nums)
    assert [max([nums[i] for i in range(n) if wl[i] >= k]) for k in range(1, n + 1)] == find_max_brute(nums)
print("Bài 3, 4: OK", window_lengths([10, 20, 50, 10]))

Bài 3, 4: OK [3, 2, 1, 4]


In [27]:
def find_minimums_of_max(nums):
    # Không gọi thẳng Solution với -nums: Solution khởi tạo best = 0 (chỉ đúng khi nums >= 0).
    # Ở đây viết lại với ngăn xếp GIẢM ngặt và best khởi tạo +vô cùng.
    n = len(nums); left, right, st = [-1] * n, [n] * n, []
    for i, x in enumerate(nums):
        while st and nums[st[-1]] <= x:
            right[st.pop()] = i
        left[i] = st[-1] if st else -1
        st.append(i)
    INF = float("inf"); best = [INF] * (n + 1)
    for i in range(n):
        L = right[i] - left[i] - 1
        best[L] = min(best[L], nums[i])
    for L in range(n - 1, 0, -1):
        best[L] = min(best[L], best[L + 1])
    return best[1:]

def wrong_negate(nums):                       # bẫy: dùng lại Solution với -nums
    return [-v for v in Solution().findMaximums([-x for x in nums])]

def min_of_max_brute(nums):
    n = len(nums)
    return [min(max(nums[l:l + k]) for l in range(n - k + 1)) for k in range(1, n + 1)]

rng = random.Random(9)
for _ in range(500):
    nums = [rng.randint(0, 9) for _ in range(rng.randint(1, 12))]
    assert find_minimums_of_max(nums) == min_of_max_brute(nums)
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK", find_minimums_of_max([0, 1, 2, 4]))
print("Bẫy đổi dấu với best=0:", wrong_negate([3, 1, 2]), "| đúng:", min_of_max_brute([3, 1, 2]))

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK [0, 1, 2, 4]
Bẫy đổi dấu với best=0: [0, 0, 0] | đúng: [1, 2, 3]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
left, right, st = [-1]*n, [n]*n, []
for i, x in enumerate(nums):
    while st and nums[st[-1]] >= x:
        right[st.pop()] = i
    left[i] = st[-1] if st else -1
    st.append(i)
best = [0] * (n + 1)
for i in range(n):
    L = right[i] - left[i] - 1
    best[L] = max(best[L], nums[i])
for L in range(n - 1, 0, -1):
    best[L] = max(best[L], best[L + 1])
return best[1:]
```

**Checklist:**
- [ ] Biên là phần tử **nhỏ hơn** gần nhất (ít nhất một phía nghiêm ngặt)?
- [ ] Ghi `best[L] = max(...)` (không ghi đè)?
- [ ] **Lan** từ độ dài lớn xuống nhỏ?
- [ ] Kiểm `[5, 5]`, `[3, 3, 3]` (giá trị trùng)?
- [ ] `O(n)`.